# Loan Applicant Default Risk Analysis & Prediction
**Author:** Dipendu Roy  
**Tools:** SQL Server, pandas, scikit-learn, Google Colab, Power BI (reporting stage)

## Business objective
Prioritise review of new loan applications using information available at application time. The dataset is synthetic: this is a portfolio demonstration, not a production credit policy.

## Data and workflow
30,000 applications comprise 19,393 historical approved loans with 12-month outcomes, 4,607 historical rejections without repayment outcomes, and 6,000 pending new applicants. Target 1 means at least one 90+ DPD snapshot in the first 12 months; unissued loans do not receive target 0.

1. Upload the two SQL exports and validate the schema.
2. Parse dates and use the fixed temporal development split.
3. Fit preprocessing and two models on training data; compare validation performance.
4. Freeze the forest and 0.20 threshold, evaluate the later test, then refit for August 2026.
5. Score and export the 6,000 pending applicants.

## Running this notebook
Run cells from top to bottom in Colab. Upload **model_development.csv** and **new_applicants_to_score.csv**, not the original clean convenience files. pandas, scikit-learn and joblib are required. Explanations precede every code cell. Existing outputs on unchanged cells are preserved from your uploaded run; edited/new cells have cleared outputs and need execution. No local rerun of your model is claimed here because your exported input CSVs were not attached.

## Interpretation limits
Labels exist only for approved issued loans; performance on rejected applicants or all applicants is not established. All data are synthetic. Calibration, fairness and monetary loss optimisation were not assessed. SQL EDA previously used all historical labels, including the later model-fitting test period. The random internal validation and later temporal test assess different settings. Preserve these qualifications in portfolio reporting.


## 01. Upload the two SQL exports

Select model_development.csv and new_applicants_to_score.csv from data/model_exports. This notebook runs in Google Colab; uploads last only for the current runtime. Keep local copies. Neither raw_practice nor the original convenience model files should be substituted.


In [2]:
from google.colab import files

uploaded = files.upload()

Saving model_development.csv to model_development.csv
Saving new_applicants_to_score.csv to new_applicants_to_score.csv


## 02. Load CSVs and verify identity

Read both exports with pandas and interpret literal NULL as missing. Expect 19,393 development rows and 6,000 new applicants; no repeated application IDs. Development has 41 columns; new applicants should have 38 after removing confirmed empty export columns. Split counts: Training 7,700, Test 3,987, Excluded 7,706.


In [3]:
import pandas as pd

development = pd.read_csv(
    "model_development.csv",
    na_values=["NULL"]
)

new_applicants = pd.read_csv(
    "new_applicants_to_score.csv",
    na_values=["NULL"]
)

print("Development data:", development.shape)
print("New applicants:", new_applicants.shape)

print("\nDataset split counts:")
print(development["Dataset_Split"].value_counts())

print("\nDuplicate application IDs:")
print("Development:", development["Application_ID"].duplicated().sum())
print("New applicants:", new_applicants["Application_ID"].duplicated().sum())

Development data: (19393, 41)
New applicants: (6000, 40)

Dataset split counts:
Dataset_Split
Excluded    7706
Training    7700
Test        3987
Name: count, dtype: int64

Duplicate application IDs:
Development: 0
New applicants: 0


## 03. Compare the export schemas

The target, observation-end date and split label are development-only metadata. Compare the remaining columns with the new-applicant file. Unexpected Unnamed columns indicate blank CSV headers, not long column names. Investigate any other schema difference before modelling.


In [4]:
# These belong only in the development file
development_only = {
    "Default_90DPD_12M",
    "Observation_End_Date",
    "Dataset_Split"
}

expected_features = set(development.columns) - development_only

print("Unexpected columns in new applicants:")
print(sorted(set(new_applicants.columns) - expected_features))

print("\nMissing columns in new applicants:")
print(sorted(expected_features - set(new_applicants.columns)))

print("\nAll new-applicant columns:")
print(new_applicants.columns.tolist())

Unexpected columns in new applicants:
['Unnamed: 38', 'Unnamed: 39']

Missing columns in new applicants:
[]

All new-applicant columns:
['Application_ID', 'Applicant_ID', 'Application_Date', 'Cohort', 'City_ID', 'City', 'State', 'City_Tier', 'Age', 'Employment_Type', 'Employment_Sector', 'Employment_Tenure_Years', 'Housing_Status', 'Dependents', 'Monthly_Income_INR', 'Monthly_Living_Expenses_INR', 'Loan_Purpose', 'Requested_Amount_INR', 'Requested_Term_Months', 'Application_Channel', 'Bureau_Snapshot_Date', 'Prior_Account_Count', 'Active_Account_Count', 'Credit_History_Months', 'Synthetic_Credit_Score', 'Total_Outstanding_INR', 'Existing_Monthly_Obligation_INR', 'Card_Utilisation_Ratio', 'Total_Late_Payments_Last12M', 'Max_Prior_DPD_Last12M', 'Enquiries_Last6M', 'Estimated_Free_Cash_Flow_INR', 'Living_Expense_Ratio', 'Existing_Obligation_Ratio', 'Requested_to_Annual_Income_Ratio', 'Outstanding_to_Annual_Income_Ratio', 'Credit_Score_Missing_Flag', 'Card_Utilisation_Missing_Flag', 'Unnam

## 04. Remove only empty export artefacts

Find Unnamed columns, verify all their cells are missing or whitespace, and remove them. Stop if any contain data. This revised cell also works when no extra columns exist and when run twice. It changes DataFrames only, not the input CSVs. Assert the expected schemas and unique IDs before proceeding.


In [ ]:
for name, df in [("Development", development), ("New applicants", new_applicants)]:
    extras = [c for c in df.columns if c.startswith("Unnamed:")]
    if extras:
        values = df[extras].replace(r"^\s*$", pd.NA, regex=True)
        print(name, "non-empty export artefacts:")
        print(values.notna().sum())
        if values.notna().any().any():
            raise ValueError(f"{name}: unexpected columns contain data; inspect them.")
        df.drop(columns=extras, inplace=True)

expected_columns = set(development.columns) - development_only
assert set(new_applicants.columns) == expected_columns, "Feature schema mismatch"
assert development.shape == (19393, 41)
assert new_applicants.shape == (6000, 38)
for df in [development, new_applicants]:
    assert df["Application_ID"].notna().all()
    assert df["Application_ID"].is_unique
assert development["Default_90DPD_12M"].isin([0, 1]).all()
print("Cleaned shapes:", development.shape, new_applicants.shape)


## 05. Inspect types and missing values

Review numeric versus categorical types and missing counts. Missing score means no prior accounts in this dataset; missing utilisation means no active card. Missing income propagates to income-based ratios. Do not fill missing values globally: imputation is learned within the training pipeline. NaN in the comparison for development-only fields means the field is absent from new applicants.


In [6]:
data_check = pd.DataFrame({
    "Development_Type": development.dtypes.astype(str),
    "Development_Missing": development.isna().sum(),
    "New_Applicants_Type": new_applicants.dtypes.astype(str),
    "New_Applicants_Missing": new_applicants.isna().sum()
})

display(data_check)

,Development_Type,Development_Missing,New_Applicants_Type,New_Applicants_Missing
Active_Account_Count,int64,0,int64,0.0
Age,int64,0,int64,0.0
Applicant_ID,object,0,object,0.0
Application_Channel,object,0,object,0.0
Application_Date,object,0,object,0.0
Application_ID,object,0,object,0.0
Bureau_Snapshot_Date,object,0,object,0.0
Card_Utilisation_Missing_Flag,int64,0,int64,0.0
Card_Utilisation_Ratio,float64,9212,float64,2756.0
City,object,0,object,0.0


## 06. Inspect date text before parsing

Display samples from all five date columns. The supplied user exports use DD-MM-YYYY (for example 01-08-2026 = 1 August). SQL native dates may export as YYYY-MM-DD on another machine; the following parser handles these two explicit layouts without month/day guessing.


In [8]:
for name, df, columns in [
    ("Development", development, [
        "Application_Date",
        "Bureau_Snapshot_Date",
        "Observation_End_Date"
    ]),
    ("New applicants", new_applicants, [
        "Application_Date",
        "Bureau_Snapshot_Date"
    ])
]:
    for column in columns:
        print(f"\n{name} — {column}")
        print(df[column].head(5).tolist())


Development — Application_Date
['01-01-2023', '01-01-2023', '01-01-2023', '01-01-2023', '01-01-2023']

Development — Bureau_Snapshot_Date
['01-01-2023', '01-01-2023', '01-01-2023', '01-01-2023', '01-01-2023']

Development — Observation_End_Date
['31-01-2024', '31-01-2024', '31-01-2024', '31-01-2024', '31-01-2024']

New applicants — Application_Date
['01-08-2026', '01-08-2026', '01-08-2026', '01-08-2026', '01-08-2026']

New applicants — Bureau_Snapshot_Date
['01-08-2026', '01-08-2026', '01-08-2026', '01-08-2026', '01-08-2026']


## 07. Convert dates with explicit formats

Parse DD-MM-YYYY or ISO YYYY-MM-DD, preserving existing datetime columns. Invalid dates raise an error. Dates remain metadata for time checks and splitting, not predictors. The revised cell is safe to rerun.


In [ ]:
def parse_project_dates(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        return series
    text = series.astype("string").str.strip()
    parsed = pd.to_datetime(text, format="%d-%m-%Y", errors="coerce")
    remaining = parsed.isna() & text.notna()
    parsed.loc[remaining] = pd.to_datetime(
        text.loc[remaining], format="%Y-%m-%d", errors="raise"
    )
    if parsed.isna().any():
        raise ValueError(f"Missing date in {series.name}")
    return parsed

for df in [development, new_applicants]:
    for column in ["Application_Date", "Bureau_Snapshot_Date"]:
        df[column] = parse_project_dates(df[column])
development["Observation_End_Date"] = parse_project_dates(
    development["Observation_End_Date"]
)
print(development[["Application_Date", "Bureau_Snapshot_Date", "Observation_End_Date"]].dtypes)
print(new_applicants[["Application_Date", "Bureau_Snapshot_Date"]].dtypes)


## 08. Apply the SQL temporal split

Select the 7,700-row training pool and 3,987-row later test. Assert that training labels were observable before 1 January 2025, and test applications are January-June 2025. Assert no shared application IDs. The 7,706 excluded loans remain available for the later final refit.


In [10]:
training_pool = development.loc[
    development["Dataset_Split"] == "Training"
].copy()

test_data = development.loc[
    development["Dataset_Split"] == "Test"
].copy()

# Verify the timing rules
assert (
    training_pool["Observation_End_Date"] < pd.Timestamp("2025-01-01")
).all()

assert (
    (test_data["Application_Date"] >= pd.Timestamp("2025-01-01"))
    & (test_data["Application_Date"] < pd.Timestamp("2025-07-01"))
).all()

# Verify no application appears in both groups
assert set(training_pool["Application_ID"]).isdisjoint(
    set(test_data["Application_ID"])
)

print("Training pool:", training_pool.shape)
print("Test set:", test_data.shape)
print("Timing and overlap checks passed.")

Training pool: (7700, 41)
Test set: (3987, 41)
Timing and overlap checks passed.


## 09. Create internal training and validation sets

Randomly allocate the matured training pool 80/20 using stratification and seed 42. Expect training 6,160 rows / 846 defaults and validation 1,540 / 211 defaults. This is internal random validation, not another temporal deployment simulation. Fit preprocessing on training only; reserve validation for model/threshold decisions.


In [11]:
from sklearn.model_selection import train_test_split

train_data, validation_data = train_test_split(
    training_pool,
    test_size=0.20,
    random_state=42,
    stratify=training_pool["Default_90DPD_12M"]
)

for name, df in [
    ("Training", train_data),
    ("Validation", validation_data)
]:
    print(
        f"{name}: {len(df):,} rows | "
        f"Defaults: {df['Default_90DPD_12M'].sum():,} | "
        f"Default rate: {df['Default_90DPD_12M'].mean():.2%}"
    )

Training: 6,160 rows | Defaults: 846 | Default rate: 13.73%
Validation: 1,540 rows | Defaults: 211 | Default rate: 13.70%


## 10. Separate predictors and target

Use Default_90DPD_12M as y. Exclude identifiers, dates, cohort, split and observation-end metadata; City_ID is redundant with City. Expected X: 32 predictors (24 numeric, 8 categorical). Keep identifiers in the original frames for attaching predictions. City, State and City_Tier are correlated representations retained from the original experiment; do not interpret their individual model effects causally.


In [12]:
target = "Default_90DPD_12M"

exclude_columns = [
    target,
    "Application_ID",
    "Applicant_ID",
    "Application_Date",
    "Bureau_Snapshot_Date",
    "Observation_End_Date",
    "Cohort",
    "Dataset_Split",
    "City_ID"  # City name already represents the location
]

feature_columns = [
    column for column in development.columns
    if column not in exclude_columns
]

X_train = train_data[feature_columns].copy()
y_train = train_data[target].copy()

X_validation = validation_data[feature_columns].copy()
y_validation = validation_data[target].copy()

X_test = test_data[feature_columns].copy()
y_test = test_data[target].copy()

X_new = new_applicants[feature_columns].copy()

numeric_features = X_train.select_dtypes(
    include="number"
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Total predictors:", len(feature_columns))
print("Numeric predictors:", len(numeric_features))
print("Categorical predictors:", len(categorical_features))
print("\nCategorical columns:", categorical_features)

Total predictors: 32
Numeric predictors: 24
Categorical predictors: 8

Categorical columns: ['City', 'State', 'City_Tier', 'Employment_Type', 'Employment_Sector', 'Housing_Status', 'Loan_Purpose', 'Application_Channel']


## 11. Define preprocessing without fitting

Numeric features: median imputation then standardisation. Categorical features: Unknown imputation then one-hot encoding; unseen categories are ignored rather than causing failure. Existing missingness flags distinguish structural absence from a typical imputed value. All transformations fit inside each model pipeline using X_train only. Scaling helps Logistic Regression; it is unnecessary but harmless for the forest here.


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Unknown"
    )),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Preprocessor defined successfully.")

Preprocessor defined successfully.


## 12. Fit the Logistic Regression baseline

Clone the unfitted preprocessor, combine it with Logistic Regression, and fit on 6,160 training rows only. max_iter=2000 allows convergence. Neither validation nor test participates in fitting. This gives a simple baseline for comparison.


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone

logistic_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

print("Logistic Regression training completed.")

Logistic Regression training completed.


## 13. Evaluate the baseline on validation

Estimate default probabilities and initially flag scores >=0.50. ROC-AUC measures ranking; average precision summarises precision-recall performance. Precision is the share of flags that are actual defaults; recall is the share of defaults caught; F1 is their harmonic mean. Recorded: AUC 0.7218, AP 0.3541, 16 of 211 defaults caught at 0.50. Confusion-matrix rows are actual, columns predicted.


In [15]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

validation_prob = logistic_model.predict_proba(X_validation)[:, 1]
validation_pred = (validation_prob >= 0.50).astype(int)

print(f"ROC-AUC: {roc_auc_score(y_validation, validation_prob):.4f}")
print(f"Average precision: {average_precision_score(y_validation, validation_prob):.4f}")
print(f"Precision: {precision_score(y_validation, validation_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_validation, validation_pred, zero_division=0):.4f}")
print(f"F1-score: {f1_score(y_validation, validation_pred, zero_division=0):.4f}")

matrix = confusion_matrix(y_validation, validation_pred, labels=[0, 1])

display(pd.DataFrame(
    matrix,
    index=["Actual: No default", "Actual: Default"],
    columns=["Predicted: No default", "Predicted: Default"]
))

ROC-AUC: 0.7218
Average precision: 0.3541
Precision: 0.5000
Recall: 0.0758
F1-score: 0.1317


,Predicted: No default,Predicted: Default
Actual: No default,1313,16
Actual: Default,195,16


## 14. Compare baseline thresholds

Explore the fixed candidate thresholds on validation only. Lower thresholds catch more defaults but increase false flags and review workload. At 0.15, 115 defaults were caught among 414 flags; at 0.25, 70 among 169. These are review flags, not approval decisions. Do not use the later test to choose a threshold.


In [16]:
threshold_results = []

for threshold in [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    predictions = (validation_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_validation, predictions, labels=[0, 1]
    ).ravel()

    threshold_results.append({
        "Threshold": threshold,
        "Applicants_Flagged": int(tp + fp),
        "Defaults_Caught": int(tp),
        "Defaults_Missed": int(fn),
        "Non_Defaults_Flagged": int(fp),
        "Precision": precision_score(
            y_validation, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_validation, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_validation, predictions, zero_division=0
        )
    })

threshold_comparison = pd.DataFrame(threshold_results)
display(threshold_comparison.round(4))

,Threshold,Applicants_Flagged,Defaults_Caught,Defaults_Missed,Non_Defaults_Flagged,Precision,Recall,F1
0,0.10,806,161,50,645,0.1998,0.7630,0.3166
1,0.15,414,115,96,299,0.2778,0.5450,0.3680
2,0.20,246,84,127,162,0.3415,0.3981,0.3676
3,0.25,169,70,141,99,0.4142,0.3318,0.3684
4,0.30,108,57,154,51,0.5278,0.2701,0.3574
5,0.40,62,33,178,29,0.5323,0.1564,0.2418
6,0.50,32,16,195,16,0.5000,0.0758,0.1317


## 15. Fit and compare Random Forest

Fit 300 trees with depth <=8 and at least 20 samples per leaf to limit complexity. Clone preprocessing to fit an independent pipeline on the same training rows. Recorded validation AUC/AP: forest 0.7320/0.3685 versus logistic 0.7218/0.3541. The gain is modest and was not tested for statistical significance.


In [17]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    ))
])

random_forest_model.fit(X_train, y_train)

rf_validation_prob = random_forest_model.predict_proba(
    X_validation
)[:, 1]

comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "ROC_AUC": roc_auc_score(y_validation, validation_prob),
        "Average_Precision": average_precision_score(
            y_validation, validation_prob
        )
    },
    {
        "Model": "Random Forest",
        "ROC_AUC": roc_auc_score(y_validation, rf_validation_prob),
        "Average_Precision": average_precision_score(
            y_validation, rf_validation_prob
        )
    }
])

display(comparison.round(4))

,Model,ROC_AUC,Average_Precision
0,Logistic Regression,0.7218,0.3541
1,Random Forest,0.7320,0.3685


## 16. Compare forest thresholds

Evaluate the same threshold grid on validation. Highest observed forest F1 is 0.3991 at 0.20: 235 flags, 89 true defaults, 146 false flags, 122 missed defaults. The project adopted highest validation F1 among the tested candidates after exploration; this was not a pre-registered rule or a cost-optimised lender policy.


In [18]:
rf_threshold_results = []

for threshold in [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    predictions = (rf_validation_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_validation, predictions, labels=[0, 1]
    ).ravel()

    rf_threshold_results.append({
        "Threshold": threshold,
        "Applicants_Flagged": int(tp + fp),
        "Defaults_Caught": int(tp),
        "Defaults_Missed": int(fn),
        "Non_Defaults_Flagged": int(fp),
        "Precision": precision_score(
            y_validation, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_validation, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_validation, predictions, zero_division=0
        )
    })

rf_threshold_comparison = pd.DataFrame(rf_threshold_results)
display(rf_threshold_comparison.round(4))

,Threshold,Applicants_Flagged,Defaults_Caught,Defaults_Missed,Non_Defaults_Flagged,Precision,Recall,F1
0,0.10,987,178,33,809,0.1803,0.8436,0.2972
1,0.15,423,122,89,301,0.2884,0.5782,0.3849
2,0.20,235,89,122,146,0.3787,0.4218,0.3991
3,0.25,130,59,152,71,0.4538,0.2796,0.3460
4,0.30,74,39,172,35,0.5270,0.1848,0.2737
5,0.40,22,15,196,7,0.6818,0.0711,0.1288
6,0.50,1,1,210,0,1.0000,0.0047,0.0094


## 17. Freeze the choice and evaluate the later test

Keep Random Forest and threshold 0.20 fixed, then evaluate the 3,987-row 2025 test. Recorded AUC 0.6932, AP 0.2683, precision 0.2908, recall 0.3295, F1 0.3089. Matrix: TN 3,051 / FP 417 / FN 348 / TP 171. The model catches about one-third of defaults. Report the decrease from validation without retuning on test. Earlier SQL EDA included these outcomes, so this is a fitting holdout rather than a wholly untouched research sample.


In [19]:
# Fixed using validation results
selected_model = random_forest_model
selected_threshold = 0.20

test_prob = selected_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= selected_threshold).astype(int)

print(f"ROC-AUC: {roc_auc_score(y_test, test_prob):.4f}")
print(
    f"Average precision: "
    f"{average_precision_score(y_test, test_prob):.4f}"
)
print(
    f"Precision: "
    f"{precision_score(y_test, test_pred, zero_division=0):.4f}"
)
print(
    f"Recall: "
    f"{recall_score(y_test, test_pred, zero_division=0):.4f}"
)
print(
    f"F1-score: "
    f"{f1_score(y_test, test_pred, zero_division=0):.4f}"
)
print(f"Applicants flagged: {test_pred.sum()} of {len(test_pred)}")

display(pd.DataFrame(
    confusion_matrix(y_test, test_pred, labels=[0, 1]),
    index=["Actual: No default", "Actual: Default"],
    columns=["Flagged: No", "Flagged: Yes"]
))

ROC-AUC: 0.6932
Average precision: 0.2683
Precision: 0.2908
Recall: 0.3295
F1-score: 0.3089
Applicants flagged: 588 of 3987


,Flagged: No,Flagged: Yes
Actual: No default,3051,417
Actual: Default,348,171


## 18. Refit for August 2026 scoring

Confirm all historical observation windows end before new applications begin. Clone the selected pipeline and refit on all 19,393 labelled loans, including the formerly excluded and test rows. This is permissible for the later scoring date. Preserve selected_model as the earlier evaluated model; its test metrics are not an independent evaluation of final_model. Retain the 0.20 threshold, whose performance may change after refitting.


In [20]:
# Confirm historical outcomes were available before new scoring begins
assert (
    development["Observation_End_Date"].max()
    < new_applicants["Application_Date"].min()
)

# Clone preserves model settings but removes the fitted state
final_model = clone(selected_model)

final_model.fit(
    development[feature_columns],
    development[target]
)

print("Final model fitted on", len(development), "historical loans.")
print("New applicants ready for scoring:", len(new_applicants))

Final model fitted on 19393 historical loans.
New applicants ready for scoring: 6000


## 19. Score new applicants and attach review flags

Generate model-estimated probabilities for 6,000 applications; retain original features/IDs for reporting. Flag scores >=0.20 for priority review. Recorded output: 1,300 priority (21.67%), 4,700 standard (78.33%); mean estimated probability 0.1538. Estimates have not been calibrated and are not observed default outcomes or automatic lending decisions.


In [21]:
new_prob = final_model.predict_proba(
    new_applicants[feature_columns]
)[:, 1]

scored_applicants = new_applicants.copy()

scored_applicants["Predicted_Default_Probability"] = new_prob
scored_applicants["Review_Flag"] = (
    new_prob >= selected_threshold
).astype(int)

scored_applicants["Review_Status"] = (
    scored_applicants["Review_Flag"]
    .map({
        1: "Priority review",
        0: "Standard review"
    })
)

assert len(scored_applicants) == 6000
assert scored_applicants["Application_ID"].is_unique
assert scored_applicants[
    "Predicted_Default_Probability"
].between(0, 1).all()

print("Applicants scored:", len(scored_applicants))

print("\nReview counts:")
print(scored_applicants["Review_Status"].value_counts())

print("\nPredicted probability summary:")
print(
    scored_applicants["Predicted_Default_Probability"]
    .describe()
    .round(4)
)

Applicants scored: 6000

Review counts:
Review_Status
Standard review    4700
Priority review    1300
Name: count, dtype: int64

Predicted probability summary:
count    6000.0000
mean        0.1538
std         0.1005
min         0.0515
25%         0.0879
50%         0.1141
75%         0.1842
max         0.6219
Name: Predicted_Default_Probability, dtype: float64


## 20. Export predictions and reusable model bundle

Save the CSV without a pandas index, and save the fitted pipeline plus exact feature order, target and review threshold using joblib. This step does not retrain. Download CSV to data/model_exports and model to models. Save this notebook in Python. Load joblib files only from trusted sources and preserve package versions for later reuse.


In [23]:
import joblib
from google.colab import files

# Predictions for Power BI
scored_applicants.to_csv(
    "new_applicant_risk_predictions.csv",
    index=False
)

# Preprocessing, model and scoring settings
model_bundle = {
    "pipeline": final_model,
    "feature_columns": feature_columns,
    "review_threshold": selected_threshold,
    "target": target
}

joblib.dump(
    model_bundle,
    "loan_applicant_risk_model.joblib"
)

files.download("new_applicant_risk_predictions.csv")
files.download("loan_applicant_risk_model.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 21. Save evaluation evidence and software versions

Export the two validation comparison tables and per-application test predictions, plus test metrics and the package versions from this runtime. These are computed from the earlier evaluated model, not final_model. Save the JSON and validation CSVs in a reports folder and test predictions in data/model_exports. The versions describe this execution environment; they were not inferred from old notebook outputs.

In [ ]:
import json
import platform
import sklearn

comparison.to_csv("validation_model_comparison.csv", index=False)
rf_threshold_comparison.to_csv("validation_forest_thresholds.csv", index=False)
test_predictions = test_data[["Application_ID", "Application_Date"]].copy()
test_predictions["Actual_Default"] = y_test.to_numpy()
test_predictions["Predicted_Default_Probability"] = test_prob
test_predictions["Review_Flag"] = test_pred
test_predictions.to_csv("test_predictions.csv", index=False)

run_summary = {
    "model": "RandomForestClassifier",
    "threshold": float(selected_threshold),
    "evaluated_training_rows": len(X_train),
    "validation_rows": len(X_validation),
    "test_rows": len(X_test),
    "final_refit_rows": len(development),
    "test_metrics": {
        "roc_auc": float(roc_auc_score(y_test, test_prob)),
        "average_precision": float(average_precision_score(y_test, test_prob)),
        "precision": float(precision_score(y_test, test_pred, zero_division=0)),
        "recall": float(recall_score(y_test, test_pred, zero_division=0)),
        "f1": float(f1_score(y_test, test_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_test, test_pred, labels=[0, 1]).tolist()
    },
    "versions": {"python": platform.python_version(), "pandas": pd.__version__,
                 "scikit_learn": sklearn.__version__, "joblib": joblib.__version__},
    "limitations": ["Synthetic data", "Approved-loan outcomes only",
                    "Uncalibrated estimates", "Prior SQL EDA included test-period labels"]
}
with open("model_run_summary.json", "w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2)
for filename in ["validation_model_comparison.csv", "validation_forest_thresholds.csv",
                 "test_predictions.csv", "model_run_summary.json"]:
    files.download(filename)


## Recorded project findings and next step

The forest performed modestly better than Logistic Regression on validation, but later-period test performance decreased. At threshold 0.20 the evaluated model caught 171 of 519 defaults (32.95% recall) among 588 flags (29.08% precision). The refitted model prioritised 1,300 of 6,000 new applicants. The mean score of 15.38% is not an observed default rate.

Power BI should distinguish historical observed default metrics from new-applicant estimated probabilities. Review flags indicate prioritisation only. Future work includes genuinely untouched evaluation data, calibration, uncertainty estimates, group-level checks and a business-defined review capacity or cost objective. Do not reuse the present test set for iterative tuning and still call it untouched.

### What was organised
Added purpose and interpretation before every code cell; retained original model settings, random seed, feature selection and threshold. Strengthened schema checks, made empty-column removal rerunnable, added explicit two-format date parsing and added evaluation/version exports. Retained uploaded outputs for unchanged cells; revised cells have no stale outputs.
